В этом ноутбуке я проверил, может ли модель сгенерировать новую json-schema посмотрев на данные. Модель сгенерировала схему, я её сохранил в `new_schema.json`, схема успешно преобразовалась в pydantic BaseModel в файле `model.py`


In [ ]:
import json

from datasets import load_from_disk
from pydantic import BaseModel, Field
from utils import ActInfo, llm

In [ ]:
dataset = load_from_disk("../data/raw")

In [ ]:
refine_prompt = """
Тебе дана json schema, описыващая структуру данных. Проанализируй примеры данных посмотри, что можно улучшить в схеме, если необходимо, добавь описание и примеры. Верни сгенерированную схему в виде текста, и добавь свои рассуждения.

Текущая json schema:
{json_schema}

Примеры данных:
{data_samples}
"""

In [ ]:
data_samples = ""

i = 0
for row in dataset["train"]:

    if len(row["text"]) > 1000:
        continue

    data_samples += f"Полный текст:\n{row["text"]}\n"
    data_samples += f"{ActInfo(**row)}\n\n"

    i += 1
    if i == 5:
        break

In [ ]:
class SchemaGuidedReasoning(BaseModel):
    reasoning: str = Field(description="Какие были внесены изменения и почему")
    refined_json_schema: str = Field(description="Код для создания обновлённой json schema")

In [ ]:
# отработало только с 3 раза для GLM-4.5
response = llm.with_structured_output(SchemaGuidedReasoning).invoke(
    refine_prompt.format(json_schema=ActInfo.model_json_schema(), data_samples=data_samples)
)

In [ ]:
print(response.reasoning)

In [ ]:
print(response.refined_json_schema)

In [ ]:
json.dump(json.loads(response.refined_json_schema), open("new_schema.json", "w", encoding="utf8"))

In [ ]:
! datamodel-codegen --input new_schema.json --input-file-type jsonschema --output-model-type pydantic_v2.BaseModel --output model.py